# A posterior from a likelihood

No dataset this time. Just a forward model and a likelihood, and we fit the posterior directly.

The forward map here is **linear** on purpose, so the exact posterior is available in closed form and we can check the flow against something real instead of squinting at a plot.
```{note}
Tiny on purpose — a couple dozen modes, a couple thousand steps, a minute or two on a CPU. It's
here to show the API, not to be impressive. Real ones live in `FuncyFlows/examples/`.
```

In [ ]:
import matplotlib.pyplot as plt
import torch

from FuncyFlows.base_measures import CosineBasis, GaussianReferenceMeasure
from FuncyFlows.transports.continuous import (ContinuousTransformation, SumField,
                                              LinearField, MatrixField, TimeBasisConditioner)
from FuncyFlows.objectives import ReverseKL
from FuncyFlows.utils.gaussian_misfit import GaussianMisfit
from FuncyFlows.diagnostics import ImportanceCorrection
from FuncyFlows.utils.train import train

torch.manual_seed(1)
DTYPE = torch.float64
M, NUM_OBS, NOISE = 24, 12, 0.1

basis = CosineBasis(M, dtype=DTYPE)
prior = GaussianReferenceMeasure(basis, alpha=0.05, power=2.0, dtype=DTYPE)

## Some data

In [ ]:
obs_points = torch.rand(NUM_OBS, 1, dtype=DTYPE)
design_obs = basis.evaluate(obs_points)                 # [NUM_OBS, M]
truth = prior.sample(1)
data = (truth @ design_obs.T)[0] + NOISE * torch.randn(NUM_OBS, dtype=DTYPE)

misfit = GaussianMisfit(lambda v: v @ design_obs.T, data, NOISE)
print("misfit at the truth:", misfit(truth).item())

## The answer we're checking against

Linear map, Gaussian prior, Gaussian noise — the posterior is Gaussian and you can just write it down. Having ground truth in a tutorial is worth a lot.

In [ ]:
precision = torch.diag(1 / prior.variances) + design_obs.T @ design_obs / NOISE ** 2
exact_cov = torch.linalg.inv(precision)
exact_mean = exact_cov @ design_obs.T @ data / NOISE ** 2
print("exact posterior std, first five modes:", exact_cov.diagonal().sqrt()[:5].tolist())

## Fit a flow by reverse KL

`ReverseKL` needs no posterior samples, only the potential. `path_gradient=True` switches on the sticking-the-landing estimator ([Vaitl et al., 2022](https://arxiv.org/abs/2206.09016)), which has much lower variance once the fit gets close — and 'once the fit gets close' is exactly where the ordinary estimator stalls out.

In [ ]:
field = SumField(
    LinearField(M, num_time_modes=4, dtype=DTYPE),
    MatrixField(TimeBasisConditioner(M, 96, num_time_modes=4, dtype=DTYPE),
                mode_scale=prior.scale),
)
flow = ContinuousTransformation(prior, field, num_steps=12)
losses = train(ReverseKL(flow, misfit, num_samples=64, path_gradient=True),
               flow.parameters(), num_steps=1500, learning_rate=3e-3)
print(f"reverse KL {sum(losses[:50]) / 50:.2f} -> {sum(losses[-50:]) / 50:.2f}")

In [ ]:
with torch.no_grad():
    draws = flow.transport(prior.sample(2000))

print("mean error   ", (draws.mean(0) - exact_mean).norm().item())
print("std  ratio   ", (draws.std(0) / exact_cov.diagonal().sqrt())[:5].tolist())

## Now grade it

`ImportanceCorrection` ([Dax et al., 2023](https://arxiv.org/abs/2210.05686)) reweights your draws by the true posterior over the flow's own density. `efficiency` near 1 means the flow nailed it; a low number is a verdict, not a warning.

This only works because the trace is exact — with a Hutchinson estimate the efficiency comes out biased *upward*, i.e. it lies to you in the flattering direction.

In [ ]:
with torch.no_grad():
    check = ImportanceCorrection(flow, draws, misfit)
print(f"efficiency   {check.efficiency:.3f}")
print(f"log evidence {check.log_evidence:.3f}")
print("reweighted mean error", (check.mean() - exact_mean).norm().item())

In [ ]:
grid = torch.linspace(0, 1, 300, dtype=DTYPE)[:, None]
design = basis.evaluate(grid)
fig, ax = plt.subplots(figsize=(7, 3))
flow_values = draws[:200] @ design.T
ax.plot(grid[:, 0], flow_values.T, color="C0", alpha=0.05, lw=1)
ax.plot(grid[:, 0], (truth @ design.T)[0], "k", lw=1.5, label="truth")
ax.plot(grid[:, 0], exact_mean @ design.T, "C3--", lw=1.5, label="exact posterior mean")
ax.scatter(obs_points[:, 0], data, color="C3", s=14, zorder=3)
ax.legend(fontsize=8)
plt.show()

---

Next: [do it once, reuse it forever](04_amortised_posterior.ipynb).